<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_07_model_analysis/seq2one/stage_07_01a_lstm_seq2one_robustness_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_07_01a -  SEQ2ONE - Modelo LSTM - Robustness testing**

# **BLOQUE DE EJECUCIÓN COMPLETO**

In [1]:
window_sizes = [90, 180]
targets = ['delta_60']
splits = ['train', 'valid', 'test']

## **1. Imports + paths**

In [2]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

## **2. Acceso a drive**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Mounted at /content/drive


## **3. Rutas de ventanas seq2one y scalers**

In [4]:
from pathlib import Path
import os

WINDOWS_SEQ2ONE_DIR = Path(
    os.environ.get("WINDOWS_SEQ2ONE_DIR", "data/windows/seq2one/")
)

SCALERS_DIR = Path(
    os.environ.get("SCALERS_DIR", "data/scaled/")
)

In [5]:
windows_paths = {}

for w in window_sizes:
    windows_paths[w] = {}

    for t in targets:
        windows_paths[w][t] = {}

        for s in splits:
            path = (
                DRIVE_DIR
                / WINDOWS_SEQ2ONE_DIR
                / f"L{w}"
                / f"windows_{t}_{s}.npz"
            )

            windows_paths[w][t][s] = path

#display(windows_paths)

#Como llamarlo:
#path_train_L60_delta = windows_paths[60]['delta_90']['train']
#print(path_train_L60_delta)

In [6]:
scalers_paths = {}
for t in targets:
  scalers_paths[t] = {}
  path = (
                DRIVE_DIR
                / SCALERS_DIR
                / f"scaler_{t}.pkl"
            )

  scalers_paths[t] = path

display(scalers_paths)

{'delta_60': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_60.pkl')}

## **4. Reproducibilidad**

In [7]:
#def set_seeds(seed: int = 42) -> None:
#    random.seed(seed)
#    np.random.seed(seed)
#    os.environ["PYTHONHASHSEED"] = str(seed)
#set_seeds(42)

## **5. Importar métricas comunes desde .py**

In [8]:
# 1) Define el root del proyecto (DEBE existir en esta misma celda)
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

# 2) Agrega el root al PYTHONPATH (antes de importar)
if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))  # insert(0) para priorizarlo

# 3) Asegura que metrics sea paquete Python
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

# 4) Invalida cachés de import (importante en Colab)
importlib.invalidate_caches()

# 5) Diagnóstico (le muestra qué ve Python)
#print("DRIVE_DIR:", DRIVE_DIR)
#print("sys.path[0]:", sys.path[0])
#print("metrics exists:", (DRIVE_DIR / "metrics").exists())
#print("metrics __init__:", (DRIVE_DIR / "metrics" / "__init__.py").exists())

# 6) Imports reales
from metrics.seq2one_metrics import compute_seq2one_metrics
#from metrics.opportunity_filter_metrics import compute_opportunity_filter_metrics

print("OK - imports metrics.*")


OK - imports metrics.*


In [9]:
#print(compute_seq2one_metrics.__doc__)

## **6. Carga de data windows**

In [10]:
def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar.
    """

    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # Usar contexto para cerrar correctamente el archivo
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")
        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # Opcional pero recomendable: copiar a memoria
        X = X.copy()
        y = y.copy()

    return X, y


In [11]:
# --------------------------------------------------
# Función común: carga scaler .pkl (sklearn)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib (ej. StandardScaler/MinMaxScaler).
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # Carga el objeto scaler.
    scaler = joblib.load(path)

    # Devuelve scaler (tipo genérico).
    return scaler

In [12]:
from typing import Any, Dict, Mapping
from pathlib import Path

def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scalers_path: Mapping[str, Path],
    splits: tuple[str, ...] = ("train", "valid", "test"),
) -> Dict[str, Any]:
    """
    Carga X/y para los splits solicitados y el scaler correspondiente
    a un (window_size, target).

    windows_paths[L][target][split] -> Path (.npz con X,y)
    scalers_path[target] -> Path (scaler)
    """

    # --------------------------
    # 1) Validaciones base
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    if target not in scalers_path:
        raise KeyError(f"target='{target}' no existe en scalers_path")

    valid_splits = {"train", "valid", "test"}
    splits_set = set(splits)
    unknown = splits_set - valid_splits
    if unknown:
        raise ValueError(f"splits inválidos: {sorted(unknown)}. Usar {sorted(valid_splits)}")

    # Validar que existan los splits solicitados para ese target
    available = set(windows_paths[window_size][target].keys())
    missing = splits_set - available
    if missing:
        raise KeyError(
            f"Faltan splits {sorted(missing)} en windows_paths[{window_size}]['{target}']. "
            f"Disponibles: {sorted(available)}"
        )

    # --------------------------
    # 2) Paths (solo los necesarios)
    # --------------------------
    split_paths: Dict[str, Path] = {sp: windows_paths[window_size][target][sp] for sp in splits}
    scaler_path = scalers_path[target]

    # --------------------------
    # 3) Carga por split
    # --------------------------
    out_splits: Dict[str, Dict[str, Any]] = {}
    out_paths: Dict[str, str] = {}

    for sp, p in split_paths.items():
        X, y = load_npz_windows(p)
        out_splits[sp] = {"X": X, "y": y}
        out_paths[sp] = str(p)

    scaler = load_scaler(scaler_path)
    out_paths["scaler"] = str(scaler_path)

    # --------------------------
    # 4) Inferir horizonte (robusto)
    # --------------------------
    try:
        horizon = int(target.split("_")[-1])
    except Exception as e:
        raise ValueError(f"No se pudo inferir horizon desde target='{target}'. Esperado sufijo '_<int>'") from e

    # --------------------------
    # 5) Retorno
    # --------------------------
    out: Dict[str, Any] = {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": out_paths,
        "scaler": scaler,
    }
    out.update(out_splits)

    return out

In [13]:
import numpy as np

def maybe_flatten_X(X: np.ndarray, *, flatten: bool) -> np.ndarray:
    """
    Si flatten=True y X es 3D (N,L,F) -> (N, L*F)
    Si flatten=False -> retorna X tal cual.
    """
    if not flatten:
        return X
    if X.ndim == 3:
        N, L, F = X.shape
        return X.reshape(N, L * F)
    if X.ndim == 2:
        return X
    raise ValueError(f"X debe ser 2D o 3D, recibí shape={X.shape}")

In [14]:
def create_bundles(
    window_size,
    targets: list,
    windows_paths=windows_paths,
    scalers_paths=scalers_paths,
    *,
    flatten_X: bool = False,
    splits: tuple[str, ...] = ("train", "valid", "test"),
    verbose_shapes: bool = True,
    ):
    bundles = []
    for t in targets:
        b = load_windows_and_scaler(
            window_size=window_size,
            target=t,
            windows_paths=windows_paths,
            scalers_path=scalers_paths,
            splits=splits,
        )

        if flatten_X:
            for sp in splits:
                b[sp]["X"] = maybe_flatten_X(b[sp]["X"], flatten=True)

        bundles.append(b)

    if verbose_shapes:
        for b in bundles:
            for sp in splits:
                print(f"H{b['horizon']} {sp.capitalize():<5}:", b[sp]["X"].shape, b[sp]["y"].shape)
            print(f"Scaler H{b['horizon']}:", type(b["scaler"]).__name__)

    return tuple(bundles)

## **7. Sanity Check**

In [15]:
from __future__ import annotations

from typing import Any, Dict, Optional, Tuple
import numpy as np


# ============================================================
# 1) Utilidades internas (inferencias y normalización)
# ============================================================

def _as_float_array(a: Any, *, name: str) -> np.ndarray:
    """Convierte a np.ndarray float64 y valida finitud."""
    arr = np.asarray(a, dtype=np.float64)
    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")
    if not np.isfinite(arr).all():
        raise ValueError(f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}")
    return arr


def _normalize_y_seq2one(y: np.ndarray, *, name: str = "y") -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).
    Acepta: (n,), (n,1). Rechaza shapes incompatibles.
    """
    if y.ndim == 1:
        return y
    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)
    raise ValueError(f"{name} shape inválido para seq2one. Se esperaba (n,) o (n,1). Recibido {y.shape}")


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiera (seq_len, n_features, mode) desde X.
    mode:
      - "3d": X=(n, seq_len, n_features)
      - "2d": X=(n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"
    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"
    raise ValueError(f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}")

In [16]:
# ============================================================
# 2) Sanity check principal (seq2one)
# ============================================================

def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: si expected_flat_dim no está, puede usarse como "d_flat" esperado.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D (d_flat esperado). Ej: 60*20=1200.
    allow_seq_inputs_take_last:
        Si y viniera como (n, seq_len) (caso raro), permite tomar y[:, -1].
        Por defecto False (recomendado).
    """
    X = _as_float_array(X, name=f"X[{split_name}]")
    y = _as_float_array(y, name=f"y[{split_name}]")

    # Normalizar y
    if y.ndim == 2 and y.shape[1] != 1 and allow_seq_inputs_take_last:
        # caso tolerante: y=(n,seq_len) -> tomar último
        y = y[:, -1]
    y = _normalize_y_seq2one(y, name=f"y[{split_name}]")

    # Inferir modo y dimensiones de X
    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    # Validaciones básicas n_samples
    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    # Validación de shapes según modo
    if mode == "3d":
        # expected_seq_len / expected_n_features
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, recibido={seq_len}. X.shape={X.shape}"
            )
        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        # Si expected_flat_dim está, valida contra eso
        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, recibido={d_flat}. X.shape={X.shape}"
            )

        # Si no hay expected_flat_dim pero sí expected_seq_len, úselo como d_flat esperado
        if expected_flat_dim is None and expected_seq_len is not None and d_flat != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_seq_len}, recibido={d_flat}. X.shape={X.shape}"
            )

        # expected_n_features no aplica en 2D
        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim (ej: 1200) o pase X en 3D."
            )

    # Validación extra: varianza de y (para detectar targets constantes)
    y_std = float(np.std(y))
    if verbose:
        info = {
            "split": split_name,
            "X_shape": tuple(X.shape),
            "y_shape": tuple(y.shape),
            "mode": mode,
            "seq_len": seq_len if mode == "3d" else None,
            "n_features": n_features if mode == "3d" else None,
            "flat_dim": int(X.shape[1]) if mode == "2d" else None,
            "y_mean": float(np.mean(y)),
            "y_std": y_std,
            "y_min": float(np.min(y)),
            "y_max": float(np.max(y)),
        }
        print(
            f"[sanity_check_seq2one] {split_name} | X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | y_std={y_std:.6f}"
        )

    return {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "y_mean": float(np.mean(y)),
        "y_std": float(np.std(y)),
        "y_min": float(np.min(y)),
        "y_max": float(np.max(y)),
    }

In [17]:
# ============================================================
# 3) Sanity checks por bundle (train/valid/test)
# ============================================================

def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "horizon": 60,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Detecta automáticamente si X es 2D o 3D, y usa TRAIN como referencia
    si no se pasan expected_*.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    # Setear esperados desde TRAIN si no se dieron
    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        # En 3D no hace falta expected_flat_dim
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        # Para evitar confusión, no usamos expected_seq_len/n_features en 2D
        expected_seq_len = expected_seq_len  # puede quedar None
        expected_n_features = None

    # Ejecutar checks
    out_tr = sanity_check_seq2one(
        X_tr, y_tr, f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_va = sanity_check_seq2one(
        X_va, y_va, f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_te = sanity_check_seq2one(
        X_te, y_te, f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    h = bundle.get("horizon", "NA")
    if verbose:
        print(f"OK {tag} (h={h})")

    return {"train": out_tr, "valid": out_va, "test": out_te, "horizon": h}


def run_sanity_checks_all_horizons_seq2one(
    bundle_60: Dict[str, Any],
    bundle_90: Dict[str, Any],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """Corre sanity checks para ambos horizontes (ej: 60 y 90)."""
    out_60 = run_sanity_checks_for_bundle_seq2one(bundle_60, tag="h60", verbose=verbose)
    out_90 = run_sanity_checks_for_bundle_seq2one(bundle_90, tag="h90", verbose=verbose)
    return {"h60": out_60, "h90": out_90}

In [18]:
#summary_delta = run_sanity_checks_all_horizons_seq2one(bundle_delta_60, bundle_delta_90)
#summary_ret = run_sanity_checks_all_horizons_seq2one(bundle_ret_60, bundle_ret_90)

## **8. Métricas ML con testing**

In [19]:
import pandas as pd

def metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    horizon: int,
    window_size: int,
    target: str,
) -> pd.DataFrame:
    """
    Convierte un dict de métricas seq2one en una fila de DataFrame.
    """

    return pd.DataFrame([{
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon_min": horizon,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics.get("R2"),
        "DA": metrics.get("DA"),
    }])

## **9. Gestión de dataset de métricas**

In [39]:
def load_seq2one_metrics_if_exists(
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_testing",
) -> pd.DataFrame:
    path = Path(base_dir) / f"seq2one_{name}_metrics.parquet"
    if path.exists():
        return pd.read_parquet(path)
    return pd.DataFrame()

In [40]:
from pathlib import Path
import pandas as pd

def save_seq2one_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_testing",
) -> Path:
    """
    Guarda un DataFrame de métricas seq2one en formato Parquet.

    Parámetros
    ----------
    df_metrics : pd.DataFrame
        DataFrame con métricas (una fila por modelo/horizonte/split).
    name : str
        Nombre identificador del archivo (ej: 'naive_valid', 'ridge_valid').
        No incluir extensión.
    base_dir : str
        Directorio base donde se almacenan todas las métricas seq2one.

    Retorna
    -------
    out_path : Path
        Ruta completa del archivo guardado.
    """
    out_dir = Path(base_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"seq2one_{name}_metrics.parquet"
    df_metrics.to_parquet(out_path, index=False)

    print(f"[OK] Métricas guardadas en: {out_path}")
    return out_path

In [41]:
import gc, torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gc.collect()
torch.cuda.empty_cache()

# **DEFINICIÓN DE MODELO**

## **10. Definición del modelo — placeholder**

### **10.1. Modelo LSTM - many to one**

**Idea básica**

El **LSTM (Long Short-Term Memory)** es una red neuronal recurrente diseñada para
modelar **dependencias temporales** en secuencias, manteniendo un estado interno
que permite recordar información relevante a lo largo del tiempo.

A diferencia del MLP, el LSTM **no aplana la ventana**, sino que procesa la
secuencia histórica **paso a paso**, preservando el orden temporal de los datos.

En configuración **many-to-one**, el modelo recibe una secuencia histórica
y produce un único valor escalar futuro.

Formalmente, el modelo puede expresarse como:

$$
h_t = \mathrm{LSTM}(x_t, h_{t-1})
$$

$$
\hat{y}_t = W_o h_T + b_o
$$

donde:
- $x_t \in \mathbb{R}^{20}$ es el vector de features en el minuto $t$,
- $h_t$ es el estado oculto del LSTM,
- $h_T$ resume toda la ventana histórica (por ejemplo, 60 minutos),
- $W_o, b_o$ son los parámetros de la capa de salida.

---

**Regularización (LSTM)**

**Riesgo:** Medio–alto, debido a la capacidad del modelo y a su memoria temporal.

La regularización **no es automática** y debe controlarse explícitamente:

- **Early stopping:**
  - Mecanismo principal para evitar sobreajuste.
- **Control del tamaño del estado oculto:**
  - Hidden size moderado.
- **Número de capas limitado:**
  - 1 (máximo 2) capas LSTM.
- **Dropout (opcional):**
  - Aplicado entre capas, no dentro de la recurrencia.

La regularización en LSTM es principalmente **estructural y temporal**, más que
puramente paramétrica.

---

**Por qué el LSTM es relevante en este proyecto**

- Entrada **secuencial explícita**: 60 × 20 (minutos × features).
- Capacidad para capturar:
  - dependencias temporales,
  - dinámica intradía,
  - patrones que no son accesibles a modelos aplanados.
- Modelo:
  - más expresivo que MLP,
  - más alineado con la naturaleza temporal del problema.

El LSTM es el **primer modelo del pipeline que explota directamente la estructura
temporal**, marcando la transición desde enfoques estáticos (seq2one aplanado)
hacia modelos verdaderamente secuenciales.

---

**Hiperparámetros iniciales**

Para este stage (sin tuning):

- Tipo: LSTM many-to-one
- Número de capas: 1
- Dimensión del estado oculto: moderada (por ejemplo, 64–128)
- Dropout: desactivado inicialmente
- Optimización: Adam
- Early stopping: activado
- **Sin validación interna automática** (la evaluación se realiza externamente en VALID)

El ajuste fino de la arquitectura y la regularización se aborda en etapas posteriores.


### **10.2. Imports (PyTorch) + semillas**

In [69]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

### **10.3. DataLoaders desde bundle (con reshape interno)**

In [70]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

def make_lstm_loaders_from_bundle_3d(
    bundle: dict,
    *,
    seq_len: int | None = None,
    n_features: int | None = None,
    batch_size: int = 16384,
    num_workers: int = 2,
) -> dict:
    """
    Crea loaders train/valid/test para LSTM many-to-one.

    Espera:
      - X: (n, seq_len, n_features)  (ya 3D)
      - y: (n,) o (n,1)  -> (n,1)

    Si seq_len/n_features se pasan, valida consistencia.
    """
    loaders = {}

    for split in ["train", "valid", "test"]:
        X = np.asarray(bundle[split]["X"], dtype=np.float32)
        y = np.asarray(bundle[split]["y"], dtype=np.float32).reshape(-1, 1)

        if X.ndim != 3:
            raise ValueError(
                f"[{split}] Se esperaba X 3D (n, seq_len, n_features). "
                f"Recibido shape={X.shape} (ndim={X.ndim})."
            )

        n, sl, nf = X.shape

        if seq_len is not None and sl != int(seq_len):
            raise ValueError(f"[{split}] seq_len esperado={seq_len}, recibido={sl}. shape={X.shape}")

        if n_features is not None and nf != int(n_features):
            raise ValueError(f"[{split}] n_features esperado={n_features}, recibido={nf}. shape={X.shape}")

        if y.shape[0] != n:
            raise ValueError(f"[{split}] X e y no alinean: X n={n}, y n={y.shape[0]}.")

        ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
        shuffle = (split == "train")

        loaders[split] = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
            drop_last=False,
        )

    return loaders


In [71]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### **10.4. Modelo LSTM many-to-one**

In [72]:
import torch
import torch.nn as nn

class LSTMSeq2One(nn.Module):
    """
    LSTM many-to-one:
      X: (B, L, F)
      y: (B, 1)
    Usa el último hidden state (h_n[-1]) como representación.
    """
    def __init__(
        self,
        *,
        n_features: int,
        hidden_size: int = 64,
        num_layers: int = 1,
        dropout: float = 0.0,   # dropout interno solo aplica si num_layers > 1
        head_dropout: float = 0.0,
        bidirectional: bool = False,
    ):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=(dropout if num_layers > 1 else 0.0),
            batch_first=True,
            bidirectional=bidirectional,
        )
        out_dim = hidden_size * (2 if bidirectional else 1)

        self.head = nn.Sequential(
            nn.Dropout(head_dropout),
            nn.Linear(out_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, L, F)
        _, (h_n, _) = self.lstm(x)
        # h_n: (num_layers * num_directions, B, hidden_size)
        h_last = h_n[-1]  # (B, hidden_size) o (B, hidden_size*2 si bidir ya está “apilado” por dirección)
        return self.head(h_last)  # (B, 1)


### **10.5. Train: early stopping + gradient clipping + scheduler**



#### **Función `eval_mse`**

In [73]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

@torch.no_grad()
def eval_mse(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    mse_sum, n = 0.0, 0
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        pred = model(xb)
        mse_sum += torch.sum((pred - yb) ** 2).item()
        n += yb.numel()
    return mse_sum / max(n, 1)

#### **Función `train_lstm`**

In [74]:
import random
import numpy as np
import torch
import torch.nn as nn

def set_global_seed(seed: int) -> None:
    """
    Fija semillas globales para reproducibilidad.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Reproducibilidad más estricta
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def train_lstm(
    loaders: dict,
    *,
    n_features: int,
    device: torch.device,
    seed: int,
    hidden_size: int = 64,
    num_layers: int = 1,
    dropout: float = 0.0,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    clip_grad_norm: float = 1.0,
    use_scheduler: bool = True,
) -> tuple[nn.Module, dict]:
    """
    Entrena un LSTM seq2one con early stopping sobre valid_mse.

    Parámetros
    ----------
    loaders : dict
        Diccionario con loaders de train/valid/test.
    n_features : int
        Número de features de entrada.
    device : torch.device
        CPU o CUDA.
    seed : int
        Semilla para reproducibilidad del entrenamiento.
    """

    # ============================================================
    # 1) Reproducibilidad
    # ============================================================
    set_global_seed(int(seed))

    # ============================================================
    # 2) Modelo
    # ============================================================
    model = LSTMSeq2One(
        n_features=n_features,
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=dropout,
    ).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    scheduler = None
    if use_scheduler:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt,
            mode="min",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
        )

    # ============================================================
    # 3) Early stopping
    # ============================================================
    best_state = None
    best_valid = float("inf")
    bad_epochs = 0

    history = {
        "seed": int(seed),
        "best_valid_mse": None,
        "epochs_ran": 0,
        "final_lr": None,
    }

    # ============================================================
    # 4) Loop de entrenamiento
    # ============================================================
    for epoch in range(1, max_epochs + 1):
        model.train()

        for xb, yb in loaders["train"]:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()

            if clip_grad_norm is not None:
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=float(clip_grad_norm),
                )

            opt.step()

        valid_mse = eval_mse(model, loaders["valid"], device)

        if scheduler is not None:
            scheduler.step(valid_mse)

        current_lr = opt.param_groups[0]["lr"]
        print(
            f"seed={seed} | epoch={epoch:02d} | "
            f"valid_mse={valid_mse:.6f} | lr={current_lr:.2e}"
        )

        history["epochs_ran"] = epoch
        history["final_lr"] = float(current_lr)

        if valid_mse < best_valid - 1e-9:
            best_valid = valid_mse
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(
                    f"seed={seed} | Early stopping (patience={patience}). "
                    f"Best valid_mse={best_valid:.6f}"
                )
                break

    # ============================================================
    # 5) Restaurar mejor estado
    # ============================================================
    history["best_valid_mse"] = float(best_valid)

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history

### **10.6. Predicción LSTM (VALID/TEST) desde X_flat**


In [75]:
@torch.no_grad()
def predict_lstm(
    model: nn.Module,
    X_seq: np.ndarray,
    *,
    device: torch.device,
    batch_size: int = 4096,
) -> np.ndarray:
    """
    Predicción para LSTM many-to-one.

    Espera:
        X_seq: (n, seq_len, n_features)
    Retorna:
        preds: (n,)
    """
    model.eval()

    if X_seq.ndim != 3:
        raise ValueError(f"Se esperaba X 3D (n, L, F). Recibido shape={X_seq.shape}")

    n = X_seq.shape[0]
    preds = []

    for i in range(0, n, batch_size):
        xb = torch.from_numpy(X_seq[i:i+batch_size]).to(device, non_blocking=True)
        yb = model(xb).squeeze(-1)
        preds.append(yb.detach().cpu().numpy())

        del xb, yb

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return np.concatenate(preds, axis=0)

In [76]:
import numpy as np

def get_metrics_torch(
    bundle: dict,
    model,
    *,
    device,
    predict_fn,
    batch_size_pred: int = 32768,
    compute_r2: bool = True,
) -> tuple[dict, dict]:
    """
    Calcula métricas valid/test para modelos seq2one.
    - predict_fn debe devolver y_pred con shape (n,) (o convertible a 1D).
    - bundle['valid'/'test']['X'] puede ser 2D (MLP) o 3D (LSTM).
    """
    # -------- VALID --------
    X_valid = np.asarray(bundle["valid"]["X"], dtype=np.float32)
    y_valid = np.asarray(bundle["valid"]["y"], dtype=np.float32).reshape(-1)

    y_pred_valid = predict_fn(model, X_valid, device=device, batch_size=batch_size_pred)
    y_pred_valid = np.asarray(y_pred_valid).reshape(-1)

    metrics_valid = compute_seq2one_metrics(y_valid, y_pred_valid, compute_r2=compute_r2)

    # -------- TEST --------
    X_test = np.asarray(bundle["test"]["X"], dtype=np.float32)
    y_test = np.asarray(bundle["test"]["y"], dtype=np.float32).reshape(-1)

    y_pred_test = predict_fn(model, X_test, device=device, batch_size=batch_size_pred)
    y_pred_test = np.asarray(y_pred_test).reshape(-1)

    metrics_test = compute_seq2one_metrics(y_test, y_pred_test, compute_r2=compute_r2)

    return metrics_valid, metrics_test

## **11. Ejecución completa**

### **Función `run_lstm`**

In [77]:
from __future__ import annotations

import gc
import time
import pandas as pd
import torch


def _ts() -> str:
    return time.strftime("%H:%M:%S")


def run_lstm(
    window_size: int,
    *,
    n_features: int = 36,
    batch_size_train: int = 4096,
    batch_size_pred: int = 128,  # <- antes 32768, demasiado alto para LSTM

    # ---- robustez ----
    seeds: int | list[int] = 42,

    # ---- hiperparámetros LSTM ----
    hidden_size: int = 64,
    num_layers: int = 1,
    dropout: float = 0.0,

    # ---- optim ----
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    clip_grad_norm: float = 1.0,
    use_scheduler: bool = True,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Ejecuta LSTM seq2one para:
      - target fijo: delta_60
      - window_size fijo: L
    Repitiendo entrenamiento para múltiples seeds (robustez).

    Retorna un DataFrame con filas valid/test por seed.
    """
    L = int(window_size)

    # normalizar seeds
    if isinstance(seeds, int):
        seeds_list = [int(seeds)]
    else:
        seeds_list = [int(s) for s in seeds]

    if verbose:
        print("\n" + "=" * 80)
        print(
            f"[{_ts()}] LSTM | SEQ2ONE | WINDOW_SIZE=L{L} | (L,F)=({L},{n_features}) "
            f"| hs={hidden_size} | layers={num_layers} | do={dropout} "
            f"| wd={weight_decay} | seeds={seeds_list}"
        )
        print("=" * 80)

    target = "delta_60"
    rows = []
    t_global = time.perf_counter()

    # ----------------------------------------------------------
    # BUILD BUNDLE (3D) una sola vez, no depende de seed
    # ----------------------------------------------------------
    bundle = None

    try:
        if verbose:
            print(f"\n[{_ts()}] [BUILD] Creando bundle (flatten_X=False) | target='{target}' | L{L} ...")
        t0 = time.perf_counter()

        (bundle,) = create_bundles(
            window_size=L,
            targets=[target],
            windows_paths=windows_paths,
            scalers_paths=scalers_paths,
            flatten_X=False,  # LSTM necesita 3D
        )

        if verbose:
            dt = time.perf_counter() - t0
            try:
                xshape = bundle["train"]["X"].shape
                yshape = bundle["train"]["y"].shape
                print(f"[{_ts()}] [BUILD] OK | train X={xshape} y={yshape} | dt={dt:.2f}s")
            except Exception:
                print(f"[{_ts()}] [BUILD] OK | dt={dt:.2f}s")

        # ======================================================
        # LOOP POR SEED
        # ======================================================
        for j, seed in enumerate(seeds_list, start=1):
            loaders = None
            model = None
            hist = None
            metrics_valid = None
            metrics_test = None
            df_v = None
            df_t = None

            try:
                if verbose:
                    print(f"\n[{_ts()}] [SEED] ({j}/{len(seeds_list)}) seed={seed}")

                # -------------------------
                # SEED (antes de loaders/modelo)
                # -------------------------
                set_global_seed(seed)

                # -------------------------
                # LOADERS
                # -------------------------
                if verbose:
                    print(f"[{_ts()}]   [LOADERS] Creando DataLoaders (3D) ...")
                t0 = time.perf_counter()

                loaders = make_lstm_loaders_from_bundle_3d(
                    bundle,
                    seq_len=L,
                    n_features=n_features,
                    batch_size=batch_size_train,
                )

                if verbose:
                    dt = time.perf_counter() - t0
                    try:
                        ntr = len(loaders["train"].dataset)
                        nva = len(loaders["valid"].dataset)
                        nte = len(loaders["test"].dataset)
                        print(f"[{_ts()}]   [LOADERS] OK | n(train/valid/test)=({ntr}/{nva}/{nte}) | dt={dt:.2f}s")
                    except Exception:
                        print(f"[{_ts()}]   [LOADERS] OK | dt={dt:.2f}s")

                # -------------------------
                # TRAIN
                # -------------------------
                if verbose:
                    print(f"[{_ts()}]   [TRAIN] Iniciando entrenamiento ...")
                t0 = time.perf_counter()

                model, hist = train_lstm(
                    loaders,
                    n_features=n_features,
                    device=device,
                    seed=seed,
                    hidden_size=hidden_size,
                    num_layers=num_layers,
                    dropout=dropout,
                    lr=lr,
                    weight_decay=weight_decay,
                    max_epochs=max_epochs,
                    patience=patience,
                    clip_grad_norm=clip_grad_norm,
                    use_scheduler=use_scheduler,
                )

                if verbose:
                    dt = time.perf_counter() - t0
                    print(f"[{_ts()}]   [TRAIN] FIN entrenamiento | dt={dt:.2f}s")

                # -------------------------
                # METRICS (valid/test)
                # -------------------------
                if verbose:
                    print(f"[{_ts()}]   [METRICS] Calculando métricas (valid/test) ...")
                t0 = time.perf_counter()

                metrics_valid, metrics_test = get_metrics_torch(
                    bundle,
                    model,
                    device=device,
                    predict_fn=predict_lstm,
                    batch_size_pred=batch_size_pred,
                    compute_r2=True,
                )

                if verbose:
                    dt = time.perf_counter() - t0
                    print(f"[{_ts()}]   [METRICS] OK (valid/test) | dt={dt:.2f}s")

                # -------------------------
                # DF APPEND
                # -------------------------
                if verbose:
                    print(f"[{_ts()}]   [DF] Agregando filas a la tabla ...")
                t0 = time.perf_counter()

                df_v = metrics_to_df(
                    metrics_valid,
                    model="lstm",
                    split="valid",
                    horizon=bundle["horizon"],
                    window_size=bundle["window_size"],
                    target=bundle["target"],
                )

                df_t = metrics_to_df(
                    metrics_test,
                    model="lstm",
                    split="test",
                    horizon=bundle["horizon"],
                    window_size=bundle["window_size"],
                    target=bundle["target"],
                )

                for df_ in (df_v, df_t):
                    # robustez
                    df_["seed"] = seed

                    # hparams
                    df_["hidden_size"] = hidden_size
                    df_["num_layers"] = num_layers
                    df_["dropout"] = dropout
                    df_["lr"] = lr
                    df_["weight_decay"] = weight_decay

                    # training info
                    if isinstance(hist, dict):
                        df_["best_valid_mse"] = hist.get("best_valid_mse")
                        df_["epochs_ran"] = hist.get("epochs_ran")
                        df_["final_lr"] = hist.get("final_lr")

                rows.append(df_v)
                rows.append(df_t)

                if verbose:
                    dt = time.perf_counter() - t0
                    print(f"[{_ts()}]   [DF] OK | dt={dt:.2f}s")

            finally:
                if verbose:
                    print(f"[{_ts()}]   [CLEAN] Liberando objetos seed={seed} ...")

                # eliminar referencias explícitamente
                try:
                    del loaders
                except Exception:
                    pass
                try:
                    del model
                except Exception:
                    pass
                try:
                    del hist
                except Exception:
                    pass
                try:
                    del metrics_valid
                except Exception:
                    pass
                try:
                    del metrics_test
                except Exception:
                    pass
                try:
                    del df_v
                except Exception:
                    pass
                try:
                    del df_t
                except Exception:
                    pass

                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
                    torch.cuda.empty_cache()

    finally:
        if verbose:
            print(f"[{_ts()}] [CLEAN] Liberando bundle ...")

        try:
            del bundle
        except Exception:
            pass

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()

    # ----------------------------------------------------------
    # FINAL DF
    # ----------------------------------------------------------
    if verbose:
        print(f"\n[{_ts()}] [FINAL] Concatenando resultados ...")
    t0 = time.perf_counter()

    if len(rows) == 0:
        return pd.DataFrame()

    df_lstm_metrics = (
        pd.concat(rows, ignore_index=True)
          .sort_values(["window_size", "target", "seed", "split", "horizon_min", "model"])
          .reset_index(drop=True)
    )

    if verbose:
        dt = time.perf_counter() - t0
        dt_all = time.perf_counter() - t_global
        print(f"[{_ts()}] [FINAL] OK | rows={len(df_lstm_metrics)} | dt_concat={dt:.2f}s | dt_total={dt_all:.2f}s")
        print(
            df_lstm_metrics[["window_size", "target", "seed", "split", "horizon_min", "model"]]
            .drop_duplicates()
            .to_string(index=False)
        )

    return df_lstm_metrics

### **Función `run_lstm_incremental`**

In [78]:
16384.0/2

8192.0

In [79]:
from pathlib import Path
import pandas as pd
import gc
import torch

def run_lstm_incremental(
    *,
    window_sizes: list[int],

    # ---- robustez ----
    seeds: int | list[int] = 42,

    # ---- hiperparámetros LSTM ----
    hidden_size: int = 64,
    num_layers: int = 1,
    dropout: float = 0.0,

    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    clip_grad_norm: float = 1.0,
    use_scheduler: bool = True,

    # ---- data ----
    n_features: int = 36,
    batch_size_train: int = 4096,
    batch_size_pred: int = 128,

    # ---- persistencia ----
    name: str = "lstm",
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Incremental para LSTM (target fijo delta_60), guardando progreso POR SEED:
      - Para cada L en window_sizes:
          - Calcula seeds faltantes (requiere valid y test por seed)
          - Ejecuta run_lstm(L, seeds=[seed]) por cada seed faltante
          - Guarda df_hist inmediatamente tras cada seed
    """

    # ------------------------------------------------------------
    # 0) Normalizar seeds
    # ------------------------------------------------------------
    if isinstance(seeds, int):
        seeds_list = [int(seeds)]
    else:
        seeds_list = [int(s) for s in seeds]

    expected_seeds = set(seeds_list)

    # ------------------------------------------------------------
    # 1) Path de métricas
    # ------------------------------------------------------------
    metrics_dir = Path("/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_testing")
    metrics_path = metrics_dir / f"seq2one_{name}_metrics.parquet"

    # ------------------------------------------------------------
    # 2) Cargar histórico
    # ------------------------------------------------------------
    if metrics_path.exists():
        df_hist = pd.read_parquet(metrics_path)
    else:
        df_hist = pd.DataFrame()

    expected_target = "delta_60"
    expected_splits = {"valid", "test"}

    # ------------------------------------------------------------
    # 3) Loop por window_size
    # ------------------------------------------------------------
    for L in window_sizes:
        L = int(L)

        # --------- determinar seeds faltantes para este L ----------
        missing_seeds = set(expected_seeds)

        if (not df_hist.empty) and {"seed", "split", "target", "window_size", "model"}.issubset(df_hist.columns):
            dfL = df_hist[
                (df_hist["model"] == name) &
                (df_hist["window_size"] == L) &
                (df_hist["target"] == expected_target) &
                (df_hist["seed"].isin(expected_seeds)) &
                (df_hist["split"].isin(expected_splits))
            ]

            # Para cada seed: splits presentes
            splits_by_seed = dfL.groupby("seed")["split"].apply(set).to_dict()
            complete_seeds = {s for s, sp in splits_by_seed.items() if expected_splits.issubset(sp)}

            missing_seeds = set(expected_seeds) - complete_seeds

        if not missing_seeds:
            if verbose:
                print(f"[SKIP] {name} L={L} {expected_target} ya existe completo para seeds={sorted(expected_seeds)}")
            continue

        if verbose:
            print(f"[RUN] {name} L={L} faltan seeds={sorted(missing_seeds)}")

        # --------- ejecutar y GUARDAR por cada seed ----------
        for seed in sorted(missing_seeds):
            if verbose:
                print(f"[RUN] {name} L={L} -> seed={seed} (guardado inmediato)")

            df_seed = None
            try:
                df_seed = run_lstm(
                    window_size=L,
                    seeds=[seed],  # <- 1 seed por corrida
                    n_features=n_features,
                    batch_size_train=batch_size_train,
                    batch_size_pred=batch_size_pred,
                    hidden_size=hidden_size,
                    num_layers=num_layers,
                    dropout=dropout,
                    lr=lr,
                    weight_decay=weight_decay,
                    max_epochs=max_epochs,
                    patience=patience,
                    clip_grad_norm=clip_grad_norm,
                    use_scheduler=use_scheduler,
                    verbose=verbose,
                )

                # asegurar model name
                df_seed["model"] = name

                # tracking de hiperparámetros por compatibilidad
                df_seed["hidden_size"] = hidden_size
                df_seed["num_layers"] = num_layers
                df_seed["dropout"] = dropout
                df_seed["lr"] = lr
                df_seed["w_decay"] = weight_decay
                df_seed["max_epochs"] = max_epochs
                df_seed["patience"] = patience
                df_seed["clip_grad_norm"] = clip_grad_norm
                df_seed["use_scheduler"] = use_scheduler

                # anexar a histórico en memoria
                if df_hist.empty:
                    df_hist = df_seed.copy()
                else:
                    df_hist = pd.concat([df_hist, df_seed], ignore_index=True)

                # quitar duplicados por seguridad
                key_cols = [
                    "model", "seed",
                    "hidden_size", "num_layers", "dropout", "lr", "w_decay",
                    "max_epochs", "patience", "clip_grad_norm", "use_scheduler",
                    "window_size", "target", "split", "horizon_min"
                ]
                keep_cols = [c for c in key_cols if c in df_hist.columns]
                if keep_cols:
                    df_hist = df_hist.drop_duplicates(subset=keep_cols, keep="last").reset_index(drop=True)

                # GUARDAR inmediatamente (checkpoint)
                save_seq2one_metrics(df_hist, name=name)

            finally:
                # liberar objetos pesados y cache GPU
                df_seed = None
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            # recargar histórico desde disco
            if metrics_path.exists():
                df_hist = pd.read_parquet(metrics_path)

    # ------------------------------------------------------------
    # 4) Salida ordenada
    # ------------------------------------------------------------
    if df_hist.empty:
        return df_hist

    sort_cols = [c for c in ["window_size", "target", "seed", "split", "horizon_min", "model"] if c in df_hist.columns]

    return (
        df_hist.sort_values(sort_cols)
               .reset_index(drop=True)
    )

### **Aplicación**

In [80]:
seeds = [
    1, 7, 42, 123, 999,
    2024, 31415, 27182, 8080, 777,
    5555, 8888, 1001, 2025, 9090,
    3333, 4444, 6666, 1212, 2121
]

In [81]:
df_lstm_all_sizes = run_lstm_incremental(
    window_sizes=[90, 180],

    # robustez
    seeds=seeds,

    # hiperparámetros
    hidden_size=64,
    num_layers=1,
    dropout=0.0,

    lr=1e-3,
    weight_decay=1e-4,
    max_epochs=30,
    patience=5,
    clip_grad_norm=1.0,
    use_scheduler=True,

    name="lstm",
    verbose=True,
)

[SKIP] lstm L=90 delta_60 ya existe completo para seeds=[1, 7, 42, 123, 777, 999, 1001, 1212, 2024, 2025, 2121, 3333, 4444, 5555, 6666, 8080, 8888, 9090, 27182, 31415]
[RUN] lstm L=180 faltan seeds=[1, 7, 42, 123, 777, 999, 1001, 1212, 2024, 2025, 2121, 3333, 4444, 5555, 6666, 8080, 8888, 9090, 27182, 31415]
[RUN] lstm L=180 -> seed=1 (guardado inmediato)

[01:08:02] LSTM | SEQ2ONE | WINDOW_SIZE=L180 | (L,F)=(180,36) | hs=64 | layers=1 | do=0.0 | wd=0.0001 | seeds=[1]

[01:08:02] [BUILD] Creando bundle (flatten_X=False) | target='delta_60' | L180 ...
H60 Train: (327972, 180, 36) (327972,)
H60 Valid: (70228, 180, 36) (70228,)
H60 Test : (70590, 180, 36) (70590,)
Scaler H60: StandardScaler
[01:08:15] [BUILD] OK | train X=(327972, 180, 36) y=(327972,) | dt=13.37s

[01:08:15] [SEED] (1/1) seed=1
[01:08:15]   [LOADERS] Creando DataLoaders (3D) ...
[01:08:26]   [LOADERS] OK | n(train/valid/test)=(327972/70228/70590) | dt=10.73s
[01:08:26]   [TRAIN] Iniciando entrenamiento ...
seed=1 | epoch=

In [82]:
df_lstm_all_sizes

,model,split,window_size,target,horizon_min,MAE,RMSE,R2,DA,seed,...,lr,weight_decay,best_valid_mse,epochs_ran,final_lr,w_decay,max_epochs,patience,clip_grad_norm,use_scheduler
0,lstm,test,90,delta_60,60,56.800622,87.257801,0.000102,0.473839,1,...,0.001,0.0001,2792.510897,6,0.00050,0.0001,30,5,1.0,True
1,lstm,valid,90,delta_60,60,37.843291,52.844213,-0.001193,0.478906,1,...,0.001,0.0001,2792.510897,6,0.00050,0.0001,30,5,1.0,True
2,lstm,test,90,delta_60,60,56.945073,87.275373,-0.000300,0.482614,7,...,0.001,0.0001,2792.443151,8,0.00050,0.0001,30,5,1.0,True
3,lstm,valid,90,delta_60,60,37.915254,52.843572,-0.001168,0.498102,7,...,0.001,0.0001,2792.443151,8,0.00050,0.0001,30,5,1.0,True
4,lstm,test,90,delta_60,60,56.785204,87.213790,0.001111,0.479329,42,...,0.001,0.0001,2792.749786,6,0.00050,0.0001,30,5,1.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,lstm,valid,180,delta_60,60,39.949627,54.345033,-0.000017,0.477017,9090,...,0.001,0.0001,2953.382579,6,0.00050,0.0001,30,5,1.0,True
76,lstm,test,180,delta_60,60,61.835531,93.244284,-0.001970,0.474044,27182,...,0.001,0.0001,2949.309467,14,0.00025,0.0001,30,5,1.0,True
77,lstm,valid,180,delta_60,60,40.015708,54.307545,0.001362,0.482440,27182,...,0.001,0.0001,2949.309467,14,0.00025,0.0001,30,5,1.0,True
78,lstm,test,180,delta_60,60,61.640629,93.138721,0.000298,0.478600,31415,...,0.001,0.0001,2951.566811,6,0.00050,0.0001,30,5,1.0,True


## **13. Análisis**

### **13.1. Análisis descriptivo comparativo por tamaño de ventana (L=90 vs L=180)**

El primer análisis (mínimo y más informativo) es un resumen estadístico por window_size y split para comparar promedio y dispersión (robustez) de las métricas.

In [83]:
metrics = ["MAE", "RMSE", "R2", "DA"]

summary = (
    df_lstm_all_sizes.groupby(["window_size", "split"])[metrics]
      .agg(["mean", "std", "min", "max", "count"])
      .round(6)
)

summary

MAE                                             RMSE  \
                        mean       std        min        max count       mean   
window_size split                                                               
90          test   56.838629  0.038739  56.785204  56.945073    20  87.254964   
            valid  37.875562  0.034487  37.817803  37.940498    20  52.848413   
180         test   61.718444  0.065135  61.640629  61.894998    20  93.144412   
            valid  39.983624  0.043666  39.922907  40.080600    20  54.332769   

                                                               R2            \
                        std        min        max count      mean       std   
window_size split                                                             
90          test   0.027185  87.197051  87.311662    20  0.000167  0.000623   
            valid  0.016374  52.815764  52.875977    20 -0.001352  0.000620   
180         test   0.040639  93.093218  93.254482    20  0.000175  0.000873   
            valid  0.012899  54.307545  54.357655    20  0.000434  0.000475   

                                                   DA                      \
                        min       max count      mean       std       min   
window_size split                                                           
90          test  -0.001132  0.001494    20  0.478003  0.004293  0.473725   
            valid -0.002397 -0.000115    20  0.482702  0.010136  0.472721   
180         test  -0.002189  0.001274    20  0.478881  0.002167  0.474044   
            valid -0.000481  0.001362    20  0.484160  0.005226  0.472308   

                                   
                        max count  
window_size split                  
90          test   0.492799    20  
            valid  0.506951    20  
180         test   0.483524    20  
            valid  0.495369    20

**Análisis descriptivo comparativo por tamaño de ventana (20 seeds)**

**1. Desempeño promedio en TEST**

- MAE  
  - L=90 → 56.84  
  - L=180 → 61.72  
  → **L=90 mejor (~5 puntos menos de error)**

- RMSE  
  - L=90 → 87.25  
  - L=180 → 93.14  
  → **L=90 mejor (~6 puntos menos de error)**

- R²  
  - L=90 → 0.00017  
  - L=180 → 0.00018  
  → Ambos ≈ 0 → **sin capacidad predictiva real (diferencia irrelevante)**

- Directional Accuracy (DA)  
  - L=90 → 0.4780  
  - L=180 → 0.4789  
  → Prácticamente iguales (sin señal direccional)

- Conclusión TEST:
  - **L=90 domina claramente en error (MAE, RMSE)**
  - R² ≈ 0 → modelo sin poder explicativo
  - DA ≈ 0.48 → comportamiento cercano al azar
  - **Aumentar la ventana no mejora el modelo**

---

**2. Robustez en TEST (desviación estándar)**

- MAE (std)  
  - L=90 → 0.0387  
  - L=180 → 0.0651  

- RMSE (std)  
  - L=90 → 0.0272  
  - L=180 → 0.0406  

- R² (std)  
  - L=90 → 0.00062  
  - L=180 → 0.00087  

- DA (std)  
  - L=90 → 0.00429  
  - L=180 → 0.00217  

- Conclusión:
  - Ambos modelos son **extremadamente estables**
  - **L=90 es más robusto en error (menor dispersión)**
  - La variabilidad es mínima → resultados altamente consistentes
  - **La estabilidad confirma que el bajo desempeño no es ruido, sino estructural**

---

**3. Desempeño promedio en VALID**

- MAE  
  - L=90 → 37.88  
  - L=180 → 39.98  

- RMSE  
  - L=90 → 52.85  
  - L=180 → 54.33  

- R²  
  - L=90 → -0.00135  
  - L=180 → 0.00043  

- DA  
  - L=90 → 0.4827  
  - L=180 → 0.4842  

- Conclusión VALID:
  - **L=90 mejor en error (MAE, RMSE)**
  - R² ≈ 0 en ambos → sin señal clara
  - DA ≈ 0.48 → comportamiento cercano a azar
  - L=180 muestra ligeras mejoras en R²/DA, pero **no son relevantes**

---

**4. Gap VALID–TEST (generalización)**

- L=90  
  - VALID R² ≈ -0.00135  
  - TEST R² ≈ 0.00017  
  - Gap ≈ 0.0015  

- L=180  
  - VALID R² ≈ 0.00043  
  - TEST R² ≈ 0.00018  
  - Gap ≈ -0.00025  

- Conclusión:
  - Gap **prácticamente nulo en ambos casos**
  - No hay overfitting
  - El modelo generaliza… pero sobre **ruido (sin señal)**

---

**5. Conclusión general**

Con 20 seeds:

- **L=90 supera consistentemente a L=180 en error**
- Mayor ventana **no aporta valor práctico**
- Ambos modelos presentan:
  - R² ≈ 0  
  - DA ≈ 0.48  
- Muy alta estabilidad entre seeds
- Gap VALID–TEST mínimo → comportamiento estadísticamente sano

---

**Conclusión final**

- El LSTM:
  - **no está capturando señal predictiva relevante**
  - se comporta cercano a un baseline aleatorio

- Aumentar la ventana (L=180):
  - **empeora el error**
  - no mejora la señal

- A diferencia del Transformer:
  - aquí no hay evidencia de que más contexto ayude
  - el modelo está en régimen de **señal débil / underfitting estructural**

→ **L=90 es la mejor elección, pero el modelo en sí no es competitivo en este setup.**

### **13.2. Test estadístico formal**

Queremos responder:

> ¿La mejora en R² en TEST al pasar de L=90 a L=180 es estadísticamente significativa?

Como usaste las mismas seeds, el test correcto es:
- Paired t-test (muestras dependientes)
- Alternativamente Wilcoxon (no paramétrico)

In [84]:
df=df_lstm_all_sizes.copy()

**1. Preparar los vectores de R² (TEST)**

In [85]:
import numpy as np

# Filtrar solo TEST
df_test = df[df["split"] == "test"]

r2_90 = (
    df_test[df_test["window_size"] == 90]
    .sort_values("seed")["R2"]
    .values
)

r2_180 = (
    df_test[df_test["window_size"] == 180]
    .sort_values("seed")["R2"]
    .values
)

r2_90, r2_180

(array([ 1.02307613e-04, -3.00453267e-04,  1.11069730e-03, -7.54338730e-05,
         1.56789386e-05,  2.07028006e-04,  2.71704732e-04,  1.80890393e-04,
         8.39077756e-04,  1.51797409e-04,  6.55486854e-04,  8.99235677e-04,
         1.49410775e-03,  4.47465747e-04,  6.31292989e-06, -4.68732900e-04,
        -5.18096223e-04, -1.13248275e-03, -5.24388235e-04, -1.73724364e-05]),
 array([ 1.16142208e-04,  1.07938147e-05,  6.66361028e-04,  1.50737227e-04,
        -2.18887513e-03,  7.03254590e-04, -5.00763659e-04, -1.93511810e-04,
         6.29065608e-04,  1.27426953e-03,  6.45758227e-04,  6.51457482e-04,
         8.02183716e-04,  9.00547020e-04,  5.47601509e-04,  1.56635336e-05,
         4.78400854e-04,  4.69867204e-04, -1.96970197e-03,  2.97696899e-04]))

**2. Paired t-test**

In [86]:
from scipy.stats import ttest_rel

t_stat, p_value = ttest_rel(r2_180, r2_90)
print("Paired t-test:")
t_stat, p_value

Paired t-test:


(np.float64(0.04147661928826824), np.float64(0.9673486918415035))

**3. Wilcoxon (más robusto con pocas muestras)**

In [87]:
from scipy.stats import wilcoxon

w_stat, p_wilcoxon = wilcoxon(r2_180, r2_90)
print("Wilcoxon:")
w_stat, p_wilcoxon

Wilcoxon:


(np.float64(93.0), np.float64(0.6742229461669922))

**Análisis estadístico formal: L=90 vs L=180 (R² en TEST, 20 seeds)**


**1. Paired t-test**

- Estadístico t = 0.0415  
- p-value = 0.9673  

Interpretación:  
- El p-value es **mucho mayor que 0.05**.  
- No se puede rechazar la hipótesis nula de igualdad de medias.

→ **No existe evidencia estadísticamente significativa** de diferencia en R² entre L=90 y L=180.

---

**2. Test no paramétrico de Wilcoxon**

- Estadístico W = 93.0  
- p-value = 0.6742  

Interpretación:  
- También **mayor que 0.05**.  
- La diferencia no es significativa incluso sin asumir normalidad.

---

**3. Conclusión estadística**

Ambos tests coinciden en que:

- La diferencia de R² entre L=90 y L=180  
- **No es estadísticamente significativa**  
- Puede explicarse completamente por variabilidad aleatoria entre seeds  

Adicionalmente:

- El estadístico t ≈ 0 indica que, en promedio,  
  **no hay diferencia sistemática entre ambas ventanas**  

---

**4. Conclusión final**

- No hay evidencia de que aumentar la ventana a L=180 mejore el desempeño.  
- L=90 y L=180 son **estadísticamente equivalentes** en R².  

Dado que:

- L=90 tiene menor error (MAE, RMSE)  
- y resultados equivalentes en R² y DA  

→ **L=90 es preferible desde el punto de vista práctico y estadístico.**

- El resultado refuerza que el problema no es la ventana,  
  sino la **falta de señal que el LSTM está logrando capturar en este setup.**

### **13.3. Evaluación del tamaño del efecto (Effect Size – Cohen’s d)**

Hasta ahora demostramos que la diferencia entre L=90 y L=180 es estadísticamente significativa (p < 0.05).

Sin embargo, la significancia estadística solo responde a la pregunta:

- ¿La diferencia existe?

No responde a la pregunta más importante desde el punto de vista práctico:

- ¿La diferencia es grande o relevante?

El tamaño del efecto (Cohen’s d) mide la magnitud real de la diferencia entre ambos modelos en relación con la variabilidad entre seeds.

En términos simples:

- Si d es pequeño → la diferencia existe, pero su impacto es débil.
- Si d es moderado → la mejora es relevante.
- Si d es grande → el cambio de ventana tiene un impacto fuerte y consistente.

Objetivo en este análisis:

Cuantificar qué tan importante es la mejora en R² al pasar de ventana 90 a ventana 180, más allá de que sea estadísticamente significativa.

In [88]:
import numpy as np

# 1) Tomar R² en TEST y alinear por seed
df_test = df[df["split"] == "test"].copy()

r2_90 = (
    df_test[df_test["window_size"] == 90]
    .sort_values("seed")["R2"]
    .to_numpy()
)

r2_180 = (
    df_test[df_test["window_size"] == 180]
    .sort_values("seed")["R2"]
    .to_numpy()
)

# 2) Diferencias pareadas (L180 - L90)
diff = r2_180 - r2_90

# 3) Cohen's d para muestras pareadas (dz): media(diff) / std(diff)
d_z = diff.mean() / diff.std(ddof=1)

# 4) Resumen útil
out = {
    "n": int(diff.size),
    "mean_R2_90": float(r2_90.mean()),
    "mean_R2_180": float(r2_180.mean()),
    "mean_diff": float(diff.mean()),
    "std_diff": float(diff.std(ddof=1)),
    "cohens_dz": float(d_z),
}

out

{'n': 20,
 'mean_R2_90': 0.00016724157072204872,
 'mean_R2_180': 0.00017534739452430004,
 'mean_diff': 8.105823802251333e-06,
 'std_diff': 0.0008739947154080873,
 'cohens_dz': 0.009274454020544673}

**Tamaño del efecto (Cohen’s d – muestras pareadas)**


**Resultados:**

- n = 20 seeds  
- R² medio L=90  = 0.00017  
- R² medio L=180 = 0.00018  
- Diferencia media = +0.000008  
- Desvío estándar de las diferencias = 0.000874  
- Cohen’s d (dz) = 0.0093  

---

**Interpretación del tamaño del efecto:**

Reglas generales para Cohen’s d:

- 0.2 → efecto pequeño  
- 0.5 → efecto moderado  
- 0.8 → efecto grande  

En este caso:

d ≈ 0.009  

Esto indica un **efecto prácticamente nulo**.

Además:

→ La diferencia es tan pequeña que es **numéricamente irrelevante**

---

**Conclusión práctica:**

- La diferencia entre L=90 y L=180:
  - Es **casi cero en magnitud**
  - No tiene impacto práctico

- El cambio de ventana:
  - **No mejora el desempeño**
  - No introduce deterioro relevante
  - Es **operativamente irrelevante**

---

**Interpretación global (significancia + efecto):**

- No existe diferencia estadísticamente significativa (p > 0.05)  
- El tamaño del efecto es **cercano a cero (d ≈ 0.01)**  
- La diferencia observada:
  - Es **ruido puro**
  - No representa ningún cambio real en el modelo  

---

**Conclusión final:**

- Aumentar la ventana a L=180:
  - **No tiene efecto medible en el desempeño**
  - No aporta valor práctico  
  - Es indistinguible de L=90 en términos estadísticos  

→ **El LSTM no se beneficia de mayor contexto temporal en este setup.**

### **13.4. Análisis de la distribución de las diferencias por seed**

Hasta ahora sabemos que:

- L=180 es mejor en promedio.
- La diferencia es estadísticamente significativa.
- El tamaño del efecto es moderado.

Pero todavía no sabemos algo clave:

> ¿La mejora ocurre de manera consistente en casi todas las seeds,
o está siendo impulsada por unas pocas inicializaciones muy favorables?

En este punto buscamos:
- Analizar la distribución de las diferencias individuales (R²_180 − R²_90).
- Ver cuántas seeds realmente mejoran.
- Evaluar si la mejora es homogénea o depende de casos extremos.

**Objetivo concreto:**

Confirmar que la superioridad de L=180 es estructural y no producto de unas pocas semillas atípicas.

In [90]:
import pandas as pd
import numpy as np

# --- Recalcular por claridad ---
df_test = df[df["split"] == "test"].copy()

r2_90 = (
    df_test[df_test["window_size"] == 90]
    .set_index("seed")["R2"]
)

r2_180 = (
    df_test[df_test["window_size"] == 180]
    .set_index("seed")["R2"]
)

df_diff = pd.DataFrame({
    "R2_90": r2_90,
    "R2_180": r2_180,
})

df_diff["diff"] = df_diff["R2_180"] - df_diff["R2_90"]

# ============================================================
# 1) Tabla ordenada
# ============================================================

df_sorted = (
    df_diff
    .sort_values("diff", ascending=False)
    .round(6)
)

print("\n=== DIFERENCIAS POR SEED (ordenado por mejora) ===")
display(df_sorted)

# ============================================================
# 2) Resumen estadístico
# ============================================================

summary = (
    df_diff["diff"]
    .agg(["count", "mean", "std", "min", "median", "max"])
    .to_frame()
    .T
    .round(6)
)

print("\n=== RESUMEN DE DIFERENCIAS (L180 - L90) ===")
display(summary)

# ============================================================
# 3) Conteo de mejoras
# ============================================================

n = len(df_diff)
n_pos = (df_diff["diff"] > 0).sum()
n_neg = (df_diff["diff"] < 0).sum()

improvement = pd.DataFrame([{
    "n_total": n,
    "n_mejora": n_pos,
    "n_empeora": n_neg,
    "pct_mejora": round(n_pos / n, 4),
    "pct_empeora": round(n_neg / n, 4),
}])

print("\n=== CONSISTENCIA DE LA MEJORA ===")
display(improvement)


=== DIFERENCIAS POR SEED (ordenado por mejora) ===


,R2_90,R2_180,diff
seed,,,
9090,-0.001132,0.000470,0.001602
2025,0.000152,0.001274,0.001122
8888,-0.000518,0.000478,0.000996
6666,0.000006,0.000548,0.000541
999,0.000207,0.000703,0.000496
8080,-0.000469,0.000016,0.000484
5555,0.000447,0.000901,0.000453
31415,-0.000017,0.000298,0.000315
7,-0.000300,0.000011,0.000311



=== RESUMEN DE DIFERENCIAS (L180 - L90) ===


,count,mean,std,min,median,max
diff,20.0,0.000008,0.000874,-0.002205,0.00012,0.001602



=== CONSISTENCIA DE LA MEJORA ===


,n_total,n_mejora,n_empeora,pct_mejora,pct_empeora
0,20,11,9,0.55,0.45


**Análisis de la distribución de las diferencias por seed (LSTM)**

**Análisis de la distribución de las diferencias por seed (LSTM)**

---

**1. Consistencia de la mejora**

- Total de seeds: 20  
- Seeds donde L=180 mejora a L=90: 11  
- Seeds donde L=180 empeora respecto a L=90: 9  

- Proporción de mejora: 55%  
- Proporción de deterioro: 45%  

**Interpretación:**

- La mejora está **casi balanceada**  
- No hay dominancia clara de ninguna ventana  
- El comportamiento es **inestable y cercano al azar**

---

**2. Magnitud de las diferencias**

- Diferencia media: +0.000008  
- Mediana: +0.00012  
- Desvío estándar: 0.000874  

- Mejor mejora observada: +0.00160  
- Peor deterioro observado: -0.00221  

**Observación clave:**

- La **media es prácticamente cero**  
- La mediana es levemente positiva, pero muy cercana a cero  
- Las diferencias son **muy pequeñas en magnitud**  
- No hay efectos grandes ni outliers dominantes  

---

**3. Estructura de los casos positivos**

- Existen mejoras en 11 seeds

Sin embargo:

- Las mejoras son **muy pequeñas**  
- También existen deterioros de magnitud similar o mayor  
- No hay un patrón claro que favorezca sistemáticamente a L=180  

---

**4. Conclusión estructural**

La comparación entre L=90 y L=180 muestra que:

- La superioridad de L=180 **NO es estructural**  
- La mejora:
  - Es inconsistente  
  - Es de magnitud despreciable  
  - No es robusta  

Por el contrario:

- La distribución está **centrada en cero**  
- El comportamiento es compatible con **ruido estadístico**

---

**Conclusión final**

- El comportamiento del LSTM indica que:

  - Aumentar la ventana a L=180:
    - **No aporta señal adicional útil**
    - No genera mejoras consistentes

- La diferencia observada:
  - Es extremadamente pequeña  
  - No es estructural  
  - No es operativamente relevante  

→ **La elección de L=180 no está justificada ni estadística ni empíricamente.**

### **13.5. Conclusión final del análisis comparativo L=90 vs L=180 (Transformer – delta_60)**

El análisis realizado es metodológicamente completo y robusto:

- Se evaluaron 20 seeds independientes.  
- Se realizó comparación descriptiva por ventana y split.  
- Se aplicaron tests estadísticos formales (t-test pareado y Wilcoxon).  
- Se calculó tamaño del efecto (Cohen’s d).  
- Se analizó la distribución de diferencias por seed.  
- Se evaluó la consistencia de la mejora.  
- Se revisó el gap de generalización (valid vs test).  

La conclusión no se basa en una observación puntual, sino en evidencia estadística y estructural consistente.

---

**Conclusión técnica:**

- La ventana L=180 **no es superior** a L=90 para el target delta_60.  

- La diferencia en R²:
  - **No es estadísticamente significativa** (p > 0.05 en ambos tests).  

- El tamaño del efecto:
  - Es **prácticamente nulo** (d ≈ 0.01)  
  - No existe impacto real del cambio de ventana  

- La consistencia de la mejora:
  - El **55% de las seeds mejora**  
  - El **45% empeora**  
  → Comportamiento **balanceado, sin dominancia clara**

- La distribución de diferencias:
  - Está **centrada en cero**  
  - Las diferencias son **muy pequeñas**  
  - No existe patrón estructural  

- No se observa overfitting:
  - El gap VALID–TEST es prácticamente nulo  
  - Pero esto se debe a **falta de señal**, no a buen desempeño  

---

**Conclusión estructural:**

- El aumento de ventana (L=180):
  - **No aporta información útil adicional**
  - No mejora el modelo
  - No genera deterioro relevante  

- La diferencia observada:
  - Es **ruido estadístico puro**
  - No es consistente ni reproducible  

---

**Conclusión final:**

El análisis puede considerarse formalmente cerrado y la evidencia indica que:

→ **L=90 y L=180 son estadísticamente equivalentes para el LSTM en este target.**

Dado que:

- L=90 presenta menor error (MAE, RMSE)  
- y no hay mejora real en R² ni DA  

→ **L=90 es la configuración preferente desde el punto de vista práctico.**

Adicionalmente:

→ **El problema principal no es la ventana, sino la falta de señal que el modelo logra capturar en este setup.**